# Natural Language Processing — Text Summarization System
**Course:** Natural Language Processing (ET5M004) — V Semester ETC  
**Student Name:** Sumit Dahiwale  
**Roll No. / BTID:** BT240017ET  
**GitHub Repository:** [sumitdahiwale06-odd/text-summarization-system](https://github.com/sumitdahiwale06-odd/text-summarization-system.git)  

---
### Project Objective
To develop an extractive text summarization pipeline in Python that accepts an input article, performs linguistic preprocessing (normalization, sentence/word tokenization, stop-word removal), calculates sentence importance scores using Word Frequency and TF-IDF metrics, ranks sentences, and restores selected sentences to chronological order to form an informative summary.

## Step 1: Importing Required Libraries
We import `nltk` for natural language processing tasks, `scikit-learn` for TF-IDF feature extraction, and `numpy` for mathematical calculations.

In [1]:
import re
import string
import collections
import math
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Ensure required NLTK corpora are downloaded
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
print("Libraries loaded successfully.")

## Step 2: Load Sample Input Article
We use a technical article discussing Artificial Intelligence and Machine Learning fundamentals.

In [2]:
raw_text = """Artificial Intelligence (AI) and Machine Learning (ML) have fundamentally transformed modern technology and daily life. Machine learning algorithms learn patterns and correlations directly from extensive datasets rather than adhering strictly to handcrafted rules. Supervised learning trains models on labeled training pairs, whereas unsupervised learning discovers latent structures in unannotated data. Natural Language Processing, a specialized subfield of artificial intelligence, enables computers to read, interpret, and generate human language effectively. Modern deep learning architectures utilize multilayer neural networks to perform complex cognitive tasks including speech recognition, computer vision, and machine translation. However, deploying these sophisticated systems requires substantial computational power, rigorous data sanitization, and continuous monitoring. Ethical considerations regarding algorithmic bias, transparency, and data privacy remain critical challenges in ongoing AI governance. As researchers continue refining these architectures, future developments promise more energy-efficient and explainable intelligent systems."""

print(f"Raw text length: {len(raw_text)} characters")
print(f"Raw word count: {len(raw_text.split())} words")

## Step 3: Sentence Tokenization (Preserving Original Sentences)
In Extractive Summarization, preserving original sentences with exact capitalization and punctuation is critical because the final summary consists of these exact sentences.

In [3]:
sentences = sent_tokenize(raw_text.strip())
print(f"Total sentences extracted: {len(sentences)}\n")
for i, s in enumerate(sentences):
    print(f"[Sentence {i+1}]: {s}")

## Step 4: Word Tokenization, Lowercasing, and Stop-Word Removal
We convert words to lowercase and strip grammatical stop words (e.g. 'and', 'the', 'is') and punctuation to extract content keywords.

In [4]:
stop_words = set(stopwords.words('english'))

cleaned_sentence_tokens = []
for s in sentences:
    words = word_tokenize(s.lower())
    # Retain only alphabetic words not in stop words
    filtered = [w for w in words if w.isalpha() and w not in stop_words and len(w) > 1]
    cleaned_sentence_tokens.append(filtered)

print("Sample cleaned tokens for Sentence 1:")
print(cleaned_sentence_tokens[0])

## Step 5: Feature Extraction & Sentence Scoring (Method A: Word Frequency)
We compute word frequencies across the document, normalize them by the maximum frequency, and score each sentence.

In [5]:
all_words = [w for sent_tokens in cleaned_sentence_tokens for w in sent_tokens]
word_freq = collections.Counter(all_words)
max_freq = max(word_freq.values())

# Normalize frequencies to (0, 1]
word_weights = {w: count / max_freq for w, count in word_freq.items()}
print("Top 5 most frequent content words:", word_freq.most_common(5))

freq_scores = []
for tokens in cleaned_sentence_tokens:
    if not tokens:
        freq_scores.append(0.0)
    else:
        raw_score = sum(word_weights.get(w, 0.0) for w in tokens)
        # Normalize by sqrt of length to avoid bias towards long sentences
        freq_scores.append(raw_score / math.sqrt(len(tokens)))

for i, score in enumerate(freq_scores):
    print(f"Sentence {i+1} Frequency Score: {score:.4f}")

## Step 6: Feature Extraction & Sentence Scoring (Method B: TF-IDF)
TF-IDF weighs terms by their uniqueness across sentences. Sentences with higher average TF-IDF values contain more salient keywords.

In [6]:
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(sentences)

# Centroid similarity calculation
doc_centroid = np.mean(tfidf_matrix.toarray(), axis=0, keepdims=True)
similarity_scores = cosine_similarity(tfidf_matrix, doc_centroid).flatten()

for i, sim in enumerate(similarity_scores):
    print(f"Sentence {i+1} TF-IDF Centroid Similarity: {sim:.4f}")

## Step 7: Sentence Ranking & Selection
We select the top 35% most important sentences, and sort them back into their original chronological order.

In [7]:
target_count = max(1, math.ceil(len(sentences) * 0.35))

# Pair index and score
scored_pairs = list(enumerate(similarity_scores))
# Rank descending by score
ranked_pairs = sorted(scored_pairs, key=lambda x: x[1], reverse=True)

print(f"Target summary length: {target_count} sentences")
print("Ranking order:", [f"S{idx+1} (score {score:.3f})" for idx, score in ranked_pairs])

# Top K sentence indices
selected_indices = set(idx for idx, score in ranked_pairs[:target_count])

# CRITICAL STEP: Restore chronological order
final_summary_sentences = [sentences[idx] for idx in range(len(sentences)) if idx in selected_indices]
final_summary = " ".join(final_summary_sentences)

print("\n--- GENERATED EXTRACTIVE SUMMARY ---")
print(final_summary)

## Step 8: Quantitative Evaluation & Statistics
We calculate the original word count, summary word count, and compression ratio.

In [8]:
orig_words = len(raw_text.split())
summ_words = len(final_summary.split())
compression_ratio = (summ_words / orig_words) * 100
reduction_pct = 100 - compression_ratio

print(f"Original Word Count   : {orig_words}")
print(f"Summary Word Count    : {summ_words}")
print(f"Compression Ratio     : {compression_ratio:.2f}%")
print(f"Information Reduction : {reduction_pct:.2f}%")
print(f"Sentences Retained    : {len(final_summary_sentences)} of {len(sentences)}")